(sec-pyerror)=
# Fehler, Logging und Debugging

Handle error types and troubleshoot (debug). For interactive reading and executing code blocks [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/hydro-informatics/jupyter-python-course/main) and find *b02-pyerror.ipynb*, or install {ref}`Python <install-python>` and {ref}`JupyterLab <jupyter>` locally.

```{admonition} Requirements
Make sure to complete the {ref}`Python introduction on data types <hi-python>` before diving into this tutorial.
```

```{admonition} Watch this section as a video
:class: tip, dropdown
<iframe width="701" height="394" src="https://www.youtube-nocookie.com/embed/vjRg0FAmpig" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>
<p>Watch this section as a video on the <a href="https://www.youtube.com/@hydroinformatics">@Hydro-Morphodynamics channel on YouTube</a>.</p>
```

## Fehler & Ausnahmen

### Fehlerarten

Irren ist menschlich und Python hilft uns, unsere Fehler zu finden, indem wir [error](https://docs.python.org/3/tutorial/errors.html) type descriptions] bereitstellen. Hier ist eine Liste der häufigsten Fehler, die beim Schreiben / Ausführen von Python-Code auftreten können:

| Error Type          | Description                                                 | Example                                                   |
|---------------------|-------------------------------------------------------------|-----------------------------------------------------------|
| `KeyError`          | A mapping key is not found.                                 | May occur when referencing a non-existing dictionary key. |
| `ImportError`       | Importing a module that does not exist                      | `import the-almighty-module`                              |
| `IndentationError`  | The code block indentation is not correct.                  | See {ref}`code style descriptions <py-indent-style>`      |
| `IndexError`        | An index outside the range of a variable is used.           | `nums = [1, 2]; print(nums[2])`                           |
| `NameError`         | An undefined variable is referenced.                        | `print(a)`                                                |
| `TypeError`         | Incompatible data types and/or operators are used together. | `"my_string" / 10`                                          |
| `ValueError`        | Right data type, but an inappropriate value.                                   | `int(input("Number:"))` answered with `t`                 |
| `ZeroDivisionError` | Division by zero.                                           | `3 / 0`                                                   |

Dennoch gibt es viele weitere Fehlertypen, die auftreten können, und Python-Entwickler pflegen umfassende Beschreibungen der eingebauten Ausnahmen auf der [Python-Dokumentations-Website](https://docs.python.org/3/library/exceptions.html)].


(try-except)=
### Ausnahmebehandlung mit `try` - `except`

`try` and `except` keywords test a code block and if it crashes, an `exception` is raised, but the script itself will not crash (unless otherwise defined in the `except` statement). The basic structure is:

```python
try:
    # code block
except ErrorType:
    print("Error / warning message.")
```

The `ErrorType` is technically not necessary (i.e., `except:` does the job, too), but adding an `ErrorType` is good practice to enable efficient debugging or making other users understand the origin of an error or warning. The following example illustrates the implementation of a `ValueError` in an interactive console script that requires users to type an integer value.

In [3]:
try:
    x = int(input("How many scoops of ice cream do you want? "))
except ValueError:
    print("What's that? sorry, please try again.")

How many scoops of ice cream do you want?  4


Die `try`-Anweisung kann auch eine `else`-Klausel haben, die nur ausgeführt wird, wenn der `try`-Block keine Ausnahme darstellt (hier mit Hilfe von Hilfsfunktionen wie `key_not_found` oder `handle_value`):

```python
try:
    value = a_dictionary[key]
except KeyError:
    return key_not_found(key)
else:
    return handle_value(value)
```

### Die `pass` Erklärung
Wenn wir anfangen, Code zu schreiben, stehen wir oft vor einer komplexen Herausforderung, die nicht alle gleichzeitig codiert werden kann. Für diese Herausforderungen hilft uns Python mit der `pass`-Anweisung, die die Implementierung von leeren (void) Codeblöcken und das Ausführen von Code inkrementell (d. H. Zum Debuggen des Codes) ermöglicht. Darüber hinaus helfen die obigen Fehlertypdefinitionen, Fehler zu verstehen, die wir in bereits geschriebenem Code gemacht haben. Um beispielsweise später zu definieren, was passiert, wenn Python in ein `NameError` läuft, kann eine `pass`-Anweisung wie folgt implementiert werden:

In [1]:
try:
    a = 5
    c = a + b # we want to define b later on with a complex formula
except NameError:
    pass # we know that we did not define b yet

```{tip}
The `pass` statement should only be temporary and it has a much broader application range, for example, in {ref}`functions <chpt-functions>` and {ref}`classes <ooc>`.
```

(logging)=
## Protokollierung

The `print()` function is useful to print variables or computation progress to the console (not too often though; printing takes time and slows down calculations). For robust reporting of errors or other important messages, however, a log file represents a better choice. So what is a log file or the act of logging? We know logbooks from discoverers or adventurers, who write down their experiences ordered by dates. Similarly, a code can write down (*log*) its "experiences", but in a file instead of a book. For this purpose, Python has the standard [*logging* library](https://docs.python.org/3/howto/logging.html). For the moment, it is sufficient to know that the *logging* library can be imported in any Python script with `import logging` (more information about packages, modules, and libraries is provided in the {ref}`sec-pypckg` section).

Der folgende Codeblock importiert das *logging*-Modul und verwendet die folgenden Keyword-Argumente, um das `logging.basicConfig` festzulegen:

* `filename="my-logfile.log"` lässt das Protokollierungsmodul Nachrichten an eine Datei namens `"my-logfile.log"` in demselben Verzeichnis schreiben, in dem das Python-Skript ausgeführt wird.
* `format="%(asctime)s - %(message)s` setzt das Protokollformat auf `YYYY-MM-DD HH:MM:SS.sss - `*Nachrichtentext* (weitere Formatoptionen sind im [Python docs](https://docs.python.org/3/howto/logging.html#displaying-the-date-time-in-messages)] aufgeführt).
* `filemode="w"` overwrites previous messages in the log file (remove this argument to append messages instead of overwriting).
* `level=logging.DEBUG` definiert den Schweregrad von Nachrichten, die in die Protokolldatei geschrieben werden, wobei `DEBUG` für Problemdiagnosen in Codes ausreichend ist; andere Schweregrade von Ereignissen sind:
    - `logging.INFO` to write all confirmation messages of events that worked as expected.
    - `logging.WARNING` (**standard**), um anzuzeigen, wann ein unerwartetes Ereignis aufgetreten ist oder wann ein Ereignis in der Zukunft einen Fehler verursachen kann (z. B. wegen unzureichendem Speicherplatz).
    - `logging.ERROR` to report serious problems that make the code crashing or not produce the desired result.
    - `logging.CRITICAL` ist ein umfassenderer Indikator für ernste Probleme, bei dem das Programm selbst möglicherweise nicht weiterlaufen kann (z. B. nicht nur der Code, sondern auch Python abstürzt).

Until here, messages are only written to the log file, but we cannot see any message in the console. To enable simultaneous logging to a log file and the console (Python terminal), use `logging.getLogger().addHandler(logging.StreamHandler())` that appends an *io* stream handler.

Um eine Nachricht an die Protokolldatei (und das Python-Terminal) zu schreiben, verwenden Sie
* `logging.debug("message")` für Codediagnosen,
* `logging.info("message")` für Fortschrittsinformationen (genau wie wir zuvor `print("message")` verwendet haben),
* `logging.warning("warning-message")` für unerwartete Ereignisdokumentation (ohne dass das Programm unterbrochen wird),
* `logging.error("error-message")` for errors that entail malfunction of the code, and
* `logging.critical("message")` for critical errors that may lead to program (Python) crashes.

*Warnung*, *Fehler* und *kritische* Nachrichten sollten in Ausnahmeerhebungen implementiert werden (siehe oben `try` - `except` Statements).

At the end of a script, logging should be stopped with `logging.shutdown()` because otherwise the log file is locked by Python and the Python Kernel needs to be stopped to make modifications (e.g., removing) to the log file.

In [2]:
import logging

logging.basicConfig(filename="my-logfile.log", format="%(asctime)s - %(message)s", filemode="w", level=logging.DEBUG)
logging.getLogger().addHandler(logging.StreamHandler())
logging.debug("This message is logged to the file.")
logging.info("Less severe information is also logged to the file.")
logging.warning("Warning messages are logged, too.")

a = "text"

try:
    logging.info(str(a**2))
except TypeError:
    logging.error("The variable is not numeric.")

# stop logging
logging.shutdown()

This message is logged to the file.


Less severe information is also logged to the file.


Warning messages are logged, too.


The variable is not numeric.


Und so sieht `my-logfile.log` aus:

```
2025-05-25 18:51:46,657 - This message is logged to the file.
2025-05-25 18:51:46,666 - Less severe information is also logged to the file.
2025-05-25 18:51:46,667 - Warning messages are logged, too.
2025-05-25 18:51:46,669 - The variable is not numeric.
```

Events can also be documented by instantiating a logger object with `logger = logging.getLogger(__name__)`. This favorable solution is recommended for advanced coding such as writing a new {ref}`(custom) Python library <sec-pypckg>`) (read more in the [Python docs on advanced logging](https://docs.python.org/3/howto/logging.html#advanced-logging-tutorial)). An example script with a more sophisticated logger is provided with the [Logger script at the course repository](https://github.com/hydro-informatics/material-py-codes/raw/master/logging/Logger.py).

## Debugging

Sobald Sie mehr oder weniger komplexen Code geschrieben haben, lautet die ultimative Frage: *Wird es laufen? * Die enttäuschende Antwort ist nein (in den meisten Fällen), aber es gibt ein Heilmittel namens * Debugging *, bei dem Bugs (Fehler) aus dem Code entfernt werden. Dennoch können große Codeblöcke ein Albtraum für das Debuggen sein, und dieser Abschnitt enthält einige Prinzipien, um den Faktor der Angst zu reduzieren, den das Debuggen mit sich bringen kann.

### Ausnahmen genau nutzen
Embrace critical code blocks precisely with `try` - `except` keywords and possible errors. This will help later to identify possible error origins.

```{tip}
Dokumentieren Sie von Anfang an selbstgeschriebene Fehlermeldungen in `except`-Anweisungen (z. B. in einem Markdown-Dokument) und erstellen Sie einen Entwickler {ref}`wiki <docs-wikis>`, der mögliche Fehlerquellen und Behebungsbeschreibungen enthält.
```

### Verwenden Sie beschreibende Variablennamen
Geben Sie Variablen, Funktionen und anderen Objekten beschreibende und aussagekräftige Namen. Abkürzungen werden immer notwendig sein, aber diese sollten beschreibend sein (z. B. Akronyme verwenden). Verwenden Sie für Variablen und Funktionen nur kleine Buchstaben. Für Klassen verwenden Sie CamelCase.

### Umgang mit Fehlern
Wenn eine Fehlermeldung ausgelöst wird, lesen Sie die Fehlermeldung gründlich und mehrmals, um den Ursprung des Fehlers zu verstehen. Fehlermeldungen geben oft das jeweilige Skript und die Zeilennummer an, an der der Fehler ausgelöst wurde. Wenn der Fehler nicht offensichtlich auf missbrauchte Datentypen oder eine der oben genannten Fehlermeldungen zurückzuführen ist (z. B. ein Fehler, der in einem externen Modul / Paket aufgetreten ist), verwenden Sie Ihre bevorzugte Suchmaschine, um den Fehler zu beheben.

### Prüfung des Outputs
Die Tatsache, dass Code läuft, bedeutet nicht von Natur aus, dass das Ergebnis (Ausgabe) die gewünschte Ausgabe ist. Führen Sie daher den Code mit Eingabeparametern aus, die eine vorhandene Ausgabe aus einer anderen Quelle liefern (z. B. manuelle Berechnung) und überprüfen Sie, ob die von Code erzeugte Ausgabe der vorhandenen (gewünschten) Ausgabe entspricht.

### Code mit einem strukturierten Ansatz
Denken Sie über die Codestruktur nach, bevor Sie mit dem Einstanzen einer Reihe von Codeblöcken und dem Speichern in einigen Python-Dateien beginnen. Struktur- und/oder Verhaltensdiagramme helfen bei der Entwicklung eines ausgeklügelten Code-Frameworks. Die Entwickler der Unified Modeling Language (UML) bieten solide Richtlinien für die Entwicklung von Struktur und Verhalten [UML diagrams](https://en.wikipedia.org/wiki/Unified_Modeling_Language#Diagrams) in Software Engineering].

```{tip}
Nehmen Sie ein Blatt Papier und einen Bleistift, bevor Sie mit der Entwicklung von Code beginnen, und skizzieren Sie, wie der Code die gewünschte Ausgabe erzeugt.
```

### Weiche Alternativen

Erklären sie ihr problem einem freund oder sprechen sie es einfach laut umzuformulieren und zu versuchen, ein problem einer anderen person zu erklären (auch wenn es nur eine imaginäre person oder eine gruppe von menschen ist), stellt oft eine problemaufnahme selbst dar.

Machen Sie einen Spaziergang, schlafen Sie über das Problem oder tun Sie andere Dinge mit geringer Gehirnbesetzung. Während Sie mit anderen Dingen zu tun haben, denkt Ihr Gehirn weiterhin über das Problem nach (Wikipedia widmete diesem sogenannten Prozess der [*Inkubation*](https://en.wikipedia.org/wiki/Incubation_(psychology)] sogar eine Seite).

## Learning Success Check-up

Machen Sie den [Lernerfolgstest für dieses Jupyter-Notebook](https://forms.gle/P36jM6vCtrnoMqeT8)].

````{admonition} Unfold QR Code
:class: tip, dropdown

```{image} ../img/qr-codes/gle-pyerror.png
```
````